# Week 3 — MoneyWise V1 Data Pipeline (Pathway A: PDF)

In [1]:
# Cell 0 — Mount Google Drive
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


## **PART A**

In [2]:
# Cell 1 — Workspace setup (RUN FIRST)
from pathlib import Path
BASE_DIR = Path("/content/drive/MyDrive/ai_engineering/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PROCESSED_DIR = BASE_DIR / "data" / "processed"
RAW_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

# Compatibility aliases for notebook examples
RAW = RAW_DIR
PROCESSED = PROCESSED_DIR
BASE = BASE_DIR
OUT = PROCESSED_DIR
print("Raw folder:", RAW_DIR)
print("Processed folder:", PROCESSED_DIR)


Raw folder: /content/drive/MyDrive/ai_engineering/week3_project/data/raw
Processed folder: /content/drive/MyDrive/ai_engineering/week3_project/data/processed


In [3]:
# Cell 3 — Verify uploaded inputs (self-contained)
from pathlib import Path
RAW = Path("/content/drive/MyDrive/ai_engineering/week3_project/data/raw")
RAW_DIR = RAW
BASE_DIR = RAW.parent.parent
PROCESSED_DIR = BASE_DIR / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
BASE = BASE_DIR
OUT = PROCESSED_DIR

files = sorted(f for f in RAW.rglob("*") if f.is_file())
print("Input count:", len(files))
for f in files:
  print(f.relative_to(RAW), "bytes:", f.stat().st_size)

assert files, "Upload at least one relevant input file before proceeding."
print("Upload verified. Select your relevant input(s) for Step A3.")

Input count: 2
FEN_NS2_ENG_Interactive_FA_LowRes.pdf bytes: 5226880
FLCC-for-Malaysian-adults_compressed.pdf bytes: 1156794
Upload verified. Select your relevant input(s) for Step A3.


## **PART B**

In [4]:
# Cell 4 — REQUIRED: Source Selection + Lineage Utilities
# Run ONCE after Cell 1 and Cell 3
from pathlib import Path
import json
import hashlib
from datetime import datetime, timezone
# Select the exact source before parsing.
# Never automatically take the first file by extension.
def select_source(relative_path=None, filename=None, suffixes=None):
  if relative_path:
    selected = RAW_DIR / relative_path
    if not selected.is_file():
      raise FileNotFoundError(
        f"Source not found: {selected}"
      )
    return selected
  matches = sorted(
    p for p in RAW_DIR.rglob("*")
    if p.is_file()
    and (filename is None or p.name == filename)
    and (suffixes is None or p.suffix.lower() in suffixes)
  )
  print(
    "Candidates:",
    [str(p.relative_to(RAW_DIR)) for p in matches]
  )
  if len(matches) != 1:
    raise ValueError(
    "Select exactly one source. "
    "Set relative_path to an exact file under RAW_DIR."
  )
  return matches[0]
def source_hash(path):
  return hashlib.sha256(
    Path(path).read_bytes()
  ).hexdigest()
def save_manifest(records):
  manifest = {
    "pipeline_version": "week3-v1",
    "generated_at_utc": datetime.now(timezone.utc).isoformat(),
    "records": records
  }
  target = BASE_DIR / "data" / "week3_manifest.json"
  target.parent.mkdir(parents=True, exist_ok=True)
  target.write_text(
    json.dumps(
      manifest,
      indent=2,
      ensure_ascii=False
    ),
    encoding="utf-8"
  )
  print("Saved lineage:", target)
  return manifest

# Confirmation messages
print("Cell 4 completed successfully.")
print("Source selection and lineage utilities are ready.")
print("Available functions:")
print("- select_source()")
print("- source_hash()")
print("- save_manifest()")

Cell 4 completed successfully.
Source selection and lineage utilities are ready.
Available functions:
- select_source()
- source_hash()
- save_manifest()


In [5]:
# CELL 4A — COMMON LINEAGE REGISTRY
# REQUIRED: Run once before selected Part B pathways
from pathlib import Path
lineage_entries = {}
def register_lineage(source, output, processing):
  source = Path(source).resolve()
  output = Path(output).resolve()

  # Validate source
  if not source.is_file():
    raise FileNotFoundError(
    f"Source file not found: {source}"
  )

  # Validate prepared output
  if not output.is_file():
    raise FileNotFoundError(
    f"Prepared output not found: {output}"
  )
  if output.stat().st_size == 0:
    raise ValueError(
    f"Prepared output is empty: {output}"
  )

  # Validate processing description
  if not isinstance(processing, str) or not processing.strip():
    raise ValueError(
    "A processing description is required."
  )
  processing = processing.strip()
  if processing.upper().startswith("REPLACE"):
    raise ValueError(
    "Replace the placeholder with the actual method."
  )

  # Ensure source and output belong to project workspace
  source.relative_to(RAW_DIR.resolve())
  output.relative_to(PROCESSED_DIR.resolve())

  # Same source/output pair is updated, not duplicated
  key = (str(source), str(output))
  lineage_entries[key] = {
    "source": source,
    "output": output,
    "processing": processing
  }
  print(
    "Registered:",
    source.name,
    "->",
    output.name
  )
print("Common lineage registry ready.")
print("Registered entries:", len(lineage_entries))

Common lineage registry ready.
Registered entries: 0


In [6]:
# Cell 5A1
# PATHWAY A1 — PDF/TXT (page-aware, sentence-aware version)
# CONDITIONAL REQUIRED: Run only for document-based projects
# Replaces the lab's whole-document splitter. Each chunk keeps its PDF page number,
# and words are never cut in half.
!pip -q install pypdf
import re
import pandas as pd
from pypdf import PdfReader

# Step 1: Select the exact source file
f = select_source(
    relative_path="FEN_NS2_ENG_Interactive_FA_LowRes.pdf"
)

CHUNK_SIZE = 500

# Step 2: Cleaning rules, chosen from the inspection output (chunks 0001, 0135, 0268)
CLEAN_CFG = {
    # running footer printed on the pages, optionally followed by chapter tab numbers (01 02 03)
    "footer_regex": r"Malaysia National Strategy for Financial Literacy 2026\s*[-–]\s*2030(?:\s+0\d)*",
    # lines made only of digits and number punctuation (infographic values)
    "number_line_regex": r"[\d\s.,%:/+-]+",
    # a page is the contents list if it contains ALL of these titles
    "contents_markers": ["Setting the Stage", "Voices to Vision", "Foundations for Change"],
}


def new_stats():
    return {
        "blank_pages": 0,
        "contents_pages_skipped": [],
        "number_lines_dropped": 0,
        "footers_removed": 0,
        "forced_word_cuts": 0,
    }


# Step 3: Page-level cleaning
def clean_page_text(raw, cfg, stats):
    kept = []
    for line in raw.splitlines():
        line = re.sub(r"[ \t]+", " ", line).strip()
        if not line:
            continue
        if re.fullmatch(cfg["number_line_regex"], line):
            stats["number_lines_dropped"] += 1
            continue
        kept.append(line)
    text = " ".join(kept)
    text, removed = re.subn(cfg["footer_regex"], " ", text)
    stats["footers_removed"] += removed
    return re.sub(r"\s+", " ", text).strip()


def is_contents_page(raw, cfg):
    return all(marker in raw for marker in cfg["contents_markers"])


# Step 4: Sentence-aware packing (never cuts inside a word)
def split_sentences(text):
    return re.split(r"(?<=[.!?])\s+", text)


def split_long(text, size, stats):
    pieces = []
    while len(text) > size:
        cut = text.rfind(" ", 0, size + 1)
        if cut <= 0:  # a single word longer than size: forced cut (rare)
            cut = size
            stats["forced_word_cuts"] += 1
        pieces.append(text[:cut].strip())
        text = text[cut:].strip()
    if text:
        pieces.append(text)
    return pieces


def pack_sentences(sentences, size, stats):
    chunks, current = [], ""
    for sentence in sentences:
        sentence = sentence.strip()
        if not sentence:
            continue
        if len(sentence) > size:
            if current:
                chunks.append(current)
                current = ""
            chunks.extend(split_long(sentence, size, stats))
        elif len(current) + len(sentence) + (1 if current else 0) <= size:
            current += (" " if current else "") + sentence
        else:
            chunks.append(current)
            current = sentence
    if current:
        chunks.append(current)
    return chunks


# Step 5: Page-by-page chunking (a chunk never spans two pages)
def chunk_pages(pages, size, cfg):
    stats = new_stats()
    records = []
    for page_no, raw in pages:
        if not raw.strip():
            stats["blank_pages"] += 1
            continue
        if is_contents_page(raw, cfg):
            stats["contents_pages_skipped"].append(page_no)
            continue
        text = clean_page_text(raw, cfg, stats)
        if not text:
            stats["blank_pages"] += 1
            continue
        for chunk in pack_sentences(split_sentences(text), size, stats):
            records.append({"text": chunk, "page": page_no})
    return records, stats


# Step 6: Extract pages from the selected PDF
reader = PdfReader(str(f))
pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
text = "\n".join(content for _, content in pages)
clean = re.sub(r"[ \t]+", " ", text).strip()
print("Source:", f.name)
print("Pages:", len(pages))
print("Extracted characters:", len(text))
print("Cleaned characters (lab cleaning):", len(clean))
assert clean, "No usable text: check extraction or OCR needs."

# Step 7: Compare two approaches on the same input
# A: whole-document fixed-size slices (the lab baseline)
fixed = [clean[i:i + CHUNK_SIZE] for i in range(0, len(clean), CHUNK_SIZE)]
fixed_cuts = sum(
    1 for i in range(CHUNK_SIZE, len(clean), CHUNK_SIZE)
    if clean[i - 1].isalnum() and clean[i].isalnum()
)
# B: page-aware, sentence-aware chunks
page_records, b_stats = chunk_pages(pages, CHUNK_SIZE, CLEAN_CFG)
assert page_records, "No chunks produced"


def summarize(name, texts, with_page, cuts):
    lengths = [len(t) for t in texts]
    return {
        "approach": name,
        "chunks": len(texts),
        "min_chars": min(lengths),
        "avg_chars": round(sum(lengths) / len(lengths), 1),
        "chunks_at_500_cap": sum(1 for n in lengths if n == CHUNK_SIZE),
        "chunks_with_page_%": round(100 * with_page / len(texts), 1),
        "mid_word_cuts": cuts,
    }


comparison = pd.DataFrame([
    summarize("A fixed-500 (whole document)", fixed, 0, fixed_cuts),
    summarize("B page-aware + sentence-aware", [r["text"] for r in page_records],
              sum(1 for r in page_records if r["page"] is not None), b_stats["forced_word_cuts"]),
])
print("\n--- Chunking Comparison ---")
print(comparison.to_string(index=False))
print("\nCleaning applied in B:", b_stats)
print("\nFixed sample:", repr(fixed[0][:150]))
print("\nPage-aware sample:", {"page": page_records[0]["page"], "text": page_records[0]["text"][:150]})
print("\nPATHWAY A1 completed successfully.")

Source: FEN_NS2_ENG_Interactive_FA_LowRes.pdf
Pages: 39
Extracted characters: 131507
Cleaned characters (lab cleaning): 131257

--- Chunking Comparison ---
                     approach  chunks  min_chars  avg_chars  chunks_at_500_cap  chunks_with_page_%  mid_word_cuts
 A fixed-500 (whole document)     263        257      499.1                262                 0.0            175
B page-aware + sentence-aware     330         20      374.7                 13               100.0              0

Cleaning applied in B: {'blank_pages': 1, 'contents_pages_skipped': [2], 'number_lines_dropped': 174, 'footers_removed': 37, 'forced_word_cuts': 0}

Fixed sample: 'Message from the Prime Minister \nForeword by Co-Chairs of Financial Education Network \nExecutive Summary \nChapter 1 Setting the Stage: Malaysia’s Fina'

Page-aware sample: {'page': 3, 'text': 'Message from the Prime Minister As Malaysia advances towards becoming a more inclusive and sustainable high-income nation, the financial well

In [7]:
# Cell 5A2
# PATHWAY A2 — REQUIRED after A1 (page-aware version)
# Choose and export traceable text chunks
import json

CHOSEN_METHOD = "page-aware-sentence"
approach = f"page-aware-sentence-char-{CHUNK_SIZE}"
PROCESSING_NOTE = (
    f"{approach}; page numbers kept; running footer, number-only lines "
    "and contents page removed"
)

# Step 1: Create traceable records (the source hash is computed once)
source_sha = source_hash(f)
records = [
    {
        "record_id": f"{f.stem}-{i:04d}",
        "text": item["text"],
        "source": str(f.relative_to(RAW_DIR)),
        "source_sha256": source_sha,
        "processing": approach,
        "page": item["page"],
    }
    for i, item in enumerate(page_records, 1)
]

# Step 2: Export prepared records
prepared_path = PROCESSED_DIR / "prepared_text.jsonl"
prepared_path.parent.mkdir(parents=True, exist_ok=True)
with prepared_path.open("w", encoding="utf-8") as output:
    for row in records:
        output.write(json.dumps(row, ensure_ascii=False) + "\n")

# Step 3: Basic validation
assert prepared_path.is_file(), "Prepared output not found"
assert prepared_path.stat().st_size > 0, "Prepared output is empty"
assert records, "No text chunks were produced"
assert all(row["text"].strip() for row in records), "One or more chunks are empty"
assert all(row["page"] is not None for row in records), "A chunk has no page number"
with prepared_path.open("r", encoding="utf-8") as file:
    exported_records = [json.loads(line) for line in file if line.strip()]
assert len(exported_records) == len(records), "Exported record count does not match"

# Step 4: Display evidence
print("PATHWAY A2 completed successfully.")
print("Selected approach:", approach)
print("Total chunks exported:", len(records))
print("Pages with chunks:", len({row["page"] for row in records}))
print("Prepared file:", prepared_path)
print("File size:", prepared_path.stat().st_size, "bytes")
print("Validation: PASS")
print("First record:", records[0])

PATHWAY A2 completed successfully.
Selected approach: page-aware-sentence-char-500
Total chunks exported: 330
Pages with chunks: 37
Prepared file: /content/drive/MyDrive/ai_engineering/week3_project/data/processed/prepared_text.jsonl
File size: 210422 bytes
Validation: PASS
First record: {'record_id': 'FEN_NS2_ENG_Interactive_FA_LowRes-0001', 'text': 'Message from the Prime Minister As Malaysia advances towards becoming a more inclusive and sustainable high-income nation, the financial wellbeing of our people must remain central to our national development. The first National Strategy for Financial Literacy 2019 – 2023, or NS1.0, laid a strong foundation. It brought the public and private sectors together under a shared aspiration: to uplift the financial knowledge and capabilities of Malaysians. The results speak for themselves.', 'source': 'FEN_NS2_ENG_Interactive_FA_LowRes.pdf', 'source_sha256': '0f36d2bebbd5fafdd5a3de3e0ba154ff10ba8b516f298d2a290e1118ff73650b', 'processing': 'page-

In [8]:
assert prepared_path.exists(), 'Prepared output missing.'
assert prepared_path.stat().st_size > 0, 'Prepared output is empty.'
print('Output:',prepared_path, 'Size:',prepared_path.stat().st_size)
print('Validation: file exists and is non-empty; verify content manually too.')

Output: /content/drive/MyDrive/ai_engineering/week3_project/data/processed/prepared_text.jsonl Size: 210422
Validation: file exists and is non-empty; verify content manually too.


In [9]:
# Register Pathway A — PDF/TXT
register_lineage(
    source=f,
    output=prepared_path,
    processing=PROCESSING_NOTE
)

Registered: FEN_NS2_ENG_Interactive_FA_LowRes.pdf -> prepared_text.jsonl


In [10]:
# CELL 6A — GENERATE COMMON LINEAGE MANIFEST
# REQUIRED: Run after all selected pathways are registered
assert lineage_entries, (
"No lineage entries registered. "
"Register at least one selected pathway first."
)
lineage_records = []

for i, entry in enumerate(lineage_entries.values(), 1):
  source = entry["source"]
  output = entry["output"]
  processing = entry["processing"]
  # Validate registered files again
  assert source.is_file(), f"Missing source: {source}"
  assert output.is_file(), f"Missing output: {output}"
  assert output.stat().st_size > 0, "Empty output"

  lineage_records.append({

    "source_id": f"S{i:02d}",
    "source_filename": str(
      source.relative_to(RAW_DIR.resolve())
    ),
    "source_sha256": source_hash(source),
    "prepared_output": str(
     output.relative_to(BASE_DIR.resolve())
    ),
    "prepared_sha256": source_hash(output),
    "processing_note": processing,
    "source_version": "Not verified",
    "access_rights": "Not verified",
    "status": "Prepared; quality review pending"
  })
# Reuse the common manifest writer from Cell 4
manifest = save_manifest(lineage_records)

print("\nCOMMON LINEAGE MANIFEST GENERATED")
print("Total lineage entries:", len(manifest["records"]))
print(
  "Saved to:",
  BASE_DIR / "data" / "week3_manifest.json"
)

Saved lineage: /content/drive/MyDrive/ai_engineering/week3_project/data/week3_manifest.json

COMMON LINEAGE MANIFEST GENERATED
Total lineage entries: 1
Saved to: /content/drive/MyDrive/ai_engineering/week3_project/data/week3_manifest.json


In [11]:
# CELL 6B — VERIFY COMMON LINEAGE MANIFEST
# REQUIRED: Run after Cell 6A
import json
manifest_path = BASE_DIR / "data" / "week3_manifest.json"
assert manifest_path.is_file(), (
  "Common lineage manifest not found."
)
with manifest_path.open("r", encoding="utf-8") as file:
  saved_manifest = json.load(file)
saved_records = saved_manifest.get("records", [])
assert saved_records, (
  "No lineage records found in saved manifest."
)
assert len(saved_records) == len(lineage_entries), (
  "Saved lineage count does not match registration."
)
for record in saved_records:
  source = RAW_DIR / record["source_filename"]
  output = BASE_DIR / record["prepared_output"]
  assert source.is_file(), f"Missing source: {source}"
  assert output.is_file(), f"Missing output: {output}"
  assert source_hash(source) == record["source_sha256"], (
    f"Source checksum mismatch: {source}"
  )
  assert source_hash(output) == record["prepared_sha256"], (
    f"Prepared output checksum mismatch: {output}"
  )
print("COMMON LINEAGE VALIDATION: PASS")
print("Verified records:", len(saved_records))
print("Manifest:", manifest_path)

COMMON LINEAGE VALIDATION: PASS
Verified records: 1
Manifest: /content/drive/MyDrive/ai_engineering/week3_project/data/week3_manifest.json


## **PART C**

In [12]:
# Cell 7 — REQUIRED: Common V1 Pipeline Utilities
# Run ONCE after completing Part B

from pathlib import Path
from datetime import datetime, timezone
import hashlib
import json

# Step 1: Define standard project paths
BASE_DIR = Path("/content/drive/MyDrive/ai_engineering/week3_project")
RAW_DIR = BASE_DIR / "data" / "raw"
PIPE_DIR = BASE_DIR / "data" / "pipeline_v1"

PIPE_DIR.mkdir(parents=True, exist_ok=True)

# Store registered pipeline runs for repeatability verification
PIPELINE_RUNS = {}

# Step 2: Calculate SHA-256 checksum
def sha256_file(path):
  return hashlib.sha256(
    Path(path).read_bytes()
  ).hexdigest()

# Step 3: Resolve an explicitly selected source
def resolve_source(relative_path, folder=False):
  source = RAW_DIR / relative_path
  if folder:
    if not source.is_dir():
      raise FileNotFoundError(
        f"Source folder not found: {source}"
      )
  else:
    if not source.is_file():
      raise FileNotFoundError(
        f"Source file not found: {source}"
      )
  return source

# Step 4: Execute the selected V1 data pipeline
def execute_pipeline(
  kind,
  source,
  inspect_fn,
  prepare_fn,
  configuration
):
  source = Path(source)
  # INGEST: Identify relevant source files
  if kind == "images":
    inputs = sorted(
      p for p in source.iterdir()
      if p.is_file()
      and p.suffix.lower() in {
        ".png", ".jpg", ".jpeg", ".webp"
      }
    )
  else:
    inputs = [source]

  if not inputs or any(not p.is_file() for p in inputs):
    raise ValueError(
      "No valid source files to ingest."
    )

  # INSPECT: Apply data-type-specific inspection
  inspection = inspect_fn(inputs, configuration)

  # Define output directory
  output_dir = PIPE_DIR / kind
  output_dir.mkdir(parents=True, exist_ok=True)

  # PREPARE / STORE: Apply selected transformation
  outputs, mappings, method = prepare_fn(
    inputs,
    output_dir,
    configuration
  )
  outputs = [Path(p) for p in outputs]

  # VALIDATE: Check prepared outputs
  if not outputs:
    raise ValueError(
      "No prepared outputs were produced."
    )
  for output in outputs:
    if not output.is_file():
      raise FileNotFoundError(
        f"Prepared output missing: {output}"
      )
  if output.stat().st_size == 0:
    raise ValueError(
      f"Prepared output is empty: {output}"
    )

  # Record basic validation evidence
  validation = {
    "output_count": len(outputs),
    "output_sha256": [
      sha256_file(p) for p in outputs
    ],
    "output_bytes": [
      p.stat().st_size for p in outputs
    ]
  }

  # RECORD LINEAGE: Create pipeline execution record
  record = {
    "pipeline_version": "week3-v1",
    "generated_at_utc": datetime.now(
      timezone.utc
    ).isoformat(),
    "data_type": kind,
    "configuration": configuration,
    "processing_method": method,
    "sources": [
      {
        "path": str(p),
        "sha256": sha256_file(p)
      }
      for p in inputs
    ],
    "outputs": [
      str(p) for p in outputs
    ],
    "source_output_mapping": mappings,
    "inspection": inspection,
    "validation": validation,
    "source_version": "Not verified",
    "access_rights": (
      "Not verified — check before sharing"
    )
  }

  # Save pipeline manifest
  manifest_path = output_dir / "pipeline_manifest.json"

  manifest_path.write_text(
    json.dumps(
      record,
      indent=2,
      ensure_ascii=False
    ),
    encoding="utf-8"
  )
  print(
    f"SUCCESS: {kind} | "
    f"Inputs: {len(inputs)} | "
    f"Outputs: {len(outputs)}"
  )
  print("Manifest:", manifest_path)
  return record

# Step 5: Register and run a selected pathway
def register_and_run(
  kind,
  source,
  inspect_fn,
  prepare_fn,
  configuration
):

  # Keep the runner for repeatability verification
  def runner():
    return execute_pipeline(
      kind,
      source,
      inspect_fn,
      prepare_fn,
      configuration
    )
  first = runner()

  PIPELINE_RUNS[kind] = {
    "first": first,
    "runner": runner
  }
  return first
print("Cell 7 ready.")
print("Run ONLY the selected Cell 8 pathway(s).")
print("Common pipeline utilities are available.")

Cell 7 ready.
Run ONLY the selected Cell 8 pathway(s).
Common pipeline utilities are available.


In [13]:
# Cell 8A — CONDITIONAL REQUIRED: PDF/TXT Projects (page-aware version)
# Requires Cell 7 and the helper functions from Cell 5A1.
!pip -q install pypdf
import json
from pypdf import PdfReader

# Step 1: Select the source and the same configuration used in Part B
SOURCE_A = resolve_source("FEN_NS2_ENG_Interactive_FA_LowRes.pdf")
CONFIG_A = {
    "chunk_size": CHUNK_SIZE,
    "method": "page-aware-sentence",
    "clean": CLEAN_CFG,
}


# Step 2: Inspect document content
def inspect_documents(inputs, cfg):
    reader = PdfReader(str(inputs[0]))
    texts = [page.extract_text() or "" for page in reader.pages]
    if not any(t.strip() for t in texts):
        raise ValueError("No extractable text found. Scanned PDFs may require OCR.")
    return {
        "source": inputs[0].name,
        "pages": len(texts),
        "blank_pages": sum(1 for t in texts if not t.strip()),
        "characters": sum(len(t) for t in texts),
    }


# Step 3: Prepare and store page-aware records
def prepare_documents(inputs, out, cfg):
    src = inputs[0]
    if cfg["method"] != "page-aware-sentence":
        raise ValueError("This pipeline supports 'page-aware-sentence' only")
    reader = PdfReader(str(src))
    pages = [(i + 1, page.extract_text() or "") for i, page in enumerate(reader.pages)]
    chunks, _ = chunk_pages(pages, int(cfg["chunk_size"]), cfg["clean"])
    if not chunks:
        raise ValueError("No text chunks produced")

    target = out / "prepared_text.jsonl"
    with target.open("w", encoding="utf-8") as handle:
        for i, item in enumerate(chunks, 1):
            handle.write(json.dumps({
                "id": i,
                "source": str(src),
                "text": item["text"],
                "page": item["page"],
            }, ensure_ascii=False) + "\n")

    mapping = [{"source": str(src), "output": str(target)}]
    method = (
        f"{cfg['method']} text chunks of {cfg['chunk_size']} characters; "
        "page numbers kept; footer, number-only lines and contents page removed"
    )
    return [target], mapping, method


# Step 4: Execute and register the pipeline
run_A = register_and_run(
    "documents",
    SOURCE_A,
    inspect_documents,
    prepare_documents,
    CONFIG_A
)
print("Cell 8A completed successfully.")

SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/drive/MyDrive/ai_engineering/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
Cell 8A completed successfully.


In [14]:
# Cell 9 — REQUIRED: Repeatability and Provenance Verification
# Run after your selected Cell 8A–8F pathway(s).

# Step 1: Ensure at least one pipeline has been registered
assert PIPELINE_RUNS, (
  "No pipeline runs registered. "
  "Run at least one selected Cell 8 pathway first."
)
print("--- Repeatability Verification ---")
# Step 2: Rerun each selected pipeline
for kind, entry in PIPELINE_RUNS.items():
  first = entry["first"]
  # Execute the same pipeline again
  second = entry["runner"]()
  # Step 3: Verify input provenance
  assert first["sources"] == second["sources"], (
    f"Source fingerprint changed for {kind}"
  )
  # Step 4: Verify prepared output fingerprints
  assert first["validation"] == second["validation"], (
    f"Prepared output changed for {kind}"
  )
  # Step 5: Verify source-to-output mapping
  assert (
    first["source_output_mapping"]
    == second["source_output_mapping"]
  ), (
    f"Source-to-output mapping changed for {kind}"
  )
  print(
    f"PASS: {kind} — source fingerprints, "
    "prepared outputs and mappings match"
  )
# Step 6: Display final verification message
print("\nAll selected pipelines passed repeatability smoke checks.")

--- Repeatability Verification ---
SUCCESS: documents | Inputs: 1 | Outputs: 1
Manifest: /content/drive/MyDrive/ai_engineering/week3_project/data/pipeline_v1/documents/pipeline_manifest.json
PASS: documents — source fingerprints, prepared outputs and mappings match

All selected pipelines passed repeatability smoke checks.


In [15]:
# Cell A3 — Inspection evidence (read-only)
!pip -q install pypdf
from pypdf import PdfReader
import json
for name in ["FEN_NS2_ENG_Interactive_FA_LowRes.pdf", "FLCC-for-Malaysian-adults_compressed.pdf"]:
    f = select_source(relative_path=name)
    reader = PdfReader(str(f))
    pages = [(i + 1, p.extract_text() or "") for i, p in enumerate(reader.pages)]
    print("=" * 60)
    print(name, "| bytes:", f.stat().st_size)
    print("pages:", len(pages), "| empty pages:", sum(1 for _, t in pages if len(t.strip()) < 20),
          "| chars:", sum(len(t) for _, t in pages))
    for pg in (1, len(pages) // 2):
        print(f"--- sample page {pg} ---")
        print(repr(pages[pg - 1][1].strip()[:300]))
rows = [json.loads(l) for l in prepared_path.open(encoding="utf-8") if l.strip()]
lengths = [len(r["text"]) for r in rows]
print("=" * 60)
print("prepared records:", len(rows), "| min/avg/max chars:", min(lengths), round(sum(lengths) / len(lengths), 1), max(lengths))
print("chunks exactly 500 chars (likely hard-split):", sum(1 for n in lengths if n == 500))
for r in (rows[len(rows) // 2], rows[-1]):
    print(r["record_id"], "|", repr(r["text"][:300]))

FEN_NS2_ENG_Interactive_FA_LowRes.pdf | bytes: 5226880
pages: 39 | empty pages: 1 | chars: 131469
--- sample page 1 ---
''
--- sample page 19 ---
'Strategy\nImprove debt \nmanagement \nknowledge and \npractices for long-term \nfinancial health\nKey initiatives:\nIncrease awareness and accessibility of debt counselling services and \nrelief programmes, while proactively destigmatising these services, to \nencourage early intervention and prevent escalat'
FLCC-for-Malaysian-adults_compressed.pdf | bytes: 1156794
pages: 56 | empty pages: 0 | chars: 77889
--- sample page 1 ---
'Empowering \nFinancial \nFreedom\nFinancial Literacy Core \nCompetencies For \nMalaysian Adults \nLayout FLCC Amended Final Printed B.indd   1Layout FLCC Amended Final Printed B.indd   1 14/02/2023   3:54 PM14/02/2023   3:54 PM'
--- sample page 28 ---
'24  \n3.1.3  Credit Score\nAwareness, Knowledge \nand Understanding Skills and Behaviour Confidence, Motivation and Attitudes\nAware that credit providers \nmay seek 

In [17]:
# Evidence cell — numbers for docs/week3_data_inventory.md and docs/week3_pipeline_decisions.md
import json
rows = [json.loads(line) for line in prepared_path.open(encoding="utf-8") if line.strip()]
lengths = [len(r["text"]) for r in rows]
pages_used = sorted({r["page"] for r in rows})
print("Prepared records:", len(rows))
print("Chars min/avg/max:", min(lengths), round(sum(lengths) / len(lengths), 1), max(lengths))
print("Chunks at the 500 cap:", sum(1 for n in lengths if n == 500))
print("Pages with chunks:", len(pages_used), "| first/last page:", pages_used[0], pages_used[-1])
print("Blank pages / contents pages skipped:", b_stats["blank_pages"], b_stats["contents_pages_skipped"])
print("Number-only lines dropped:", b_stats["number_lines_dropped"], "| footers removed:", b_stats["footers_removed"])
print("Forced mid-word cuts:", b_stats["forced_word_cuts"])
print("Chunks still containing the footer text:", sum("2026 - 2030" in r["text"] for r in rows))
pipe_file = PIPE_DIR / "documents" / "prepared_text.jsonl"
print("Part C output matches Part B record count:", sum(1 for _ in pipe_file.open(encoding="utf-8")) == len(rows))
print("\n--- Three spot checks ---")
for r in (rows[0], rows[len(rows) // 2], rows[-1]):
    print(r["record_id"], "| page", r["page"], "| chars", len(r["text"]))
    print("  ", repr(r["text"][:300]))

Prepared records: 330
Chars min/avg/max: 20 374.7 500
Chunks at the 500 cap: 13
Pages with chunks: 37 | first/last page: 3 39
Blank pages / contents pages skipped: 1 [2]
Number-only lines dropped: 174 | footers removed: 37
Forced mid-word cuts: 0
Chunks still containing the footer text: 0
Part C output matches Part B record count: True

--- Three spot checks ---
FEN_NS2_ENG_Interactive_FA_LowRes-0001 | page 3 | chars 487
   'Message from the Prime Minister As Malaysia advances towards becoming a more inclusive and sustainable high-income nation, the financial wellbeing of our people must remain central to our national development. The first National Strategy for Financial Literacy 2019 – 2023, or NS1.0, laid a strong fo'
FEN_NS2_ENG_Interactive_FA_LowRes-0166 | page 18 | chars 423
   'Fostering these habits is key to ensuring sustainable access to and usage of financing and requires adaptive strategies and coordination across multiple sectors. It is also crucial for these efforts to be